In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Define a simple encoder
class Encoder(nn.Module):
    def __init__(self, input_dim=28*28, hidden_dim=128):
        super(Encoder, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2)
        )

    def forward(self, x):
        return self.model(x)

# Contrastive loss function (NT-Xent Loss from SimCLR)
def contrastive_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    batch_size = z1.shape[0]
    labels = torch.arange(batch_size).to(z1.device)
    similarities = torch.mm(z1, z2.T) / temperature

    return F.cross_entropy(similarities, labels)

# Data augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.Lambda(lambda x: x.view(-1))
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Initialize model and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Encoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    for batch in dataloader:
        images, _ = batch
        images = images.to(device)

        # Generate two augmented views
        aug1 = images + 0.01 * torch.randn_like(images)  # Slight noise as augmentation
        aug2 = images + 0.01 * torch.randn_like(images)

        # Encode views
        z1 = model(aug1)
        z2 = model(aug2)

        # Compute contrastive loss
        loss = contrastive_loss(z1, z2)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

print("Training complete.")
